### ETL Silver — Financial Transactions Dataset

This notebook reads the bronze tables, cleans and enriches them,
and saves the results as Delta tables in the silver schema.

In [0]:
transactions_bronze = spark.read.table("jarvis_databricks.bronze.transactions_data_bronze")
cards_bronze = spark.read.table("jarvis_databricks.bronze.cards_data_bronze")
users_bronze = spark.read.table("jarvis_databricks.bronze.users_data_bronze")
mcc_bronze = spark.read.table("jarvis_databricks.bronze.mcc_codes_bronze")
fraud_bronze = spark.read.table("jarvis_databricks.bronze.train_fraud_labels_bronze")

display(transactions_bronze.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
9099414,2011-02-01T12:04:00.000Z,1276,2009,117.4600,Swipe Transaction,48919,Chicago,IL,60618.0,5311,null
9099415,2011-02-01T12:04:00.000Z,1370,4957,126.0400,Swipe Transaction,40078,Elk Grove,CA,95624.0,5300,null
9099416,2011-02-01T12:04:00.000Z,1645,5878,101.6700,Swipe Transaction,38602,Cleveland,OH,44121.0,5311,null
9099419,2011-02-01T12:05:00.000Z,89,2639,15.0800,Swipe Transaction,44678,Mesilla Park,NM,88047.0,5812,null
9099420,2011-02-01T12:05:00.000Z,634,2272,15.4000,Swipe Transaction,29988,Elk Grove,CA,95758.0,5411,null


In [0]:
transactions_bronze.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)



In [0]:
mcc_bronze.printSchema()

root
 |-- mcc_code: string (nullable = true)
 |-- mcc_description: string (nullable = true)



In [0]:
fraud_bronze.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)



In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, dayofweek, date_format, hour, when, coalesce, lit

# Clean and add date/time breakdown columns
transactions_clean = (
    transactions_bronze
    .withColumn("mcc", col("mcc").cast("string"))
    .withColumn("id", col("id").cast("string"))
    .withColumn("transaction_year", year("date"))
    .withColumn("transaction_month", month("date"))
    .withColumn("transaction_day", dayofmonth("date"))
    .withColumn("day_of_week_num", dayofweek("date"))
    .withColumn("day_of_week", date_format("date", "EEEE"))
    .withColumn("hour_of_day", hour("date"))
    .withColumn(
        "time_of_day",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
         .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
         .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
         .otherwise("Night")
    )
    .dropDuplicates(["id"])
)

In [0]:
# Join with mcc_codes and fraud_labels, treat missing fraud labels as False
transactions_silver = (
    transactions_clean
    .join(mcc_bronze, transactions_clean.mcc == mcc_bronze.mcc_code, "left")
    .join(fraud_bronze, transactions_clean.id == fraud_bronze.transaction_id, "left")
    .drop("mcc_code", "transaction_id")
    .withColumn("is_fraud", coalesce(col("is_fraud"), lit(False)))
)

display(transactions_silver.limit(10))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_year,transaction_month,transaction_day,day_of_week_num,day_of_week,hour_of_day,time_of_day,mcc_description,is_fraud
9103224,2011-02-02T10:23:00.000Z,1228,4332,57.0000,Swipe Transaction,96049,Austin,TX,78705.0,5541,null,2011,2,2,4,Wednesday,10,Morning,Service Stations,false
9101654,2011-02-01T22:58:00.000Z,57,6006,13.2900,Swipe Transaction,87531,Lovington,NM,88260.0,5813,null,2011,2,1,3,Tuesday,22,Night,Drinking Places (Alcoholic Beverages),false
9101514,2011-02-01T21:48:00.000Z,1699,5593,48.8900,Swipe Transaction,11546,San Jose,CA,95134.0,5813,null,2011,2,1,3,Tuesday,21,Night,Drinking Places (Alcoholic Beverages),false
9103424,2011-02-02T11:04:00.000Z,793,4234,0.5400,Swipe Transaction,50783,Fort Lauderdale,FL,33310.0,5411,null,2011,2,2,4,Wednesday,11,Morning,"Grocery Stores, Supermarkets",false
9102955,2011-02-02T09:28:00.000Z,563,1060,3.5400,Swipe Transaction,94434,Cochranville,PA,19330.0,5411,null,2011,2,2,4,Wednesday,9,Morning,"Grocery Stores, Supermarkets",false
9099831,2011-02-01T13:19:00.000Z,1963,3317,28.7000,Swipe Transaction,88852,Vacaville,CA,95687.0,4121,null,2011,2,1,3,Tuesday,13,Afternoon,Taxicabs and Limousines,false
9101807,2011-02-02T02:31:00.000Z,847,4531,225.1800,Online Transaction,96529,ONLINE,null,null,4899,null,2011,2,2,4,Wednesday,2,Night,"Cable, Satellite, and Other Pay Television Services",false
9103316,2011-02-02T10:41:00.000Z,1776,4938,-53.0000,Swipe Transaction,61195,Nashport,OH,43830.0,5541,null,2011,2,2,4,Wednesday,10,Morning,Service Stations,false
9101751,2011-02-02T01:05:00.000Z,1823,5177,73.7000,Swipe Transaction,31893,Chandler,AZ,85226.0,5311,null,2011,2,2,4,Wednesday,1,Night,Department Stores,false
9102597,2011-02-02T08:24:00.000Z,719,5518,61.0000,Swipe Transaction,22204,Rockford,IL,61107.0,5541,null,2011,2,2,4,Wednesday,8,Morning,Service Stations,false


In [0]:
cards_bronze.printSchema()
display(cards_bronze.limit(5))

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,true,2,33900.0000,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,true,1,11600.0000,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,true,1,19948.0000,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,true,2,16400.0000,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,true,2,19439.0000,01/1997,2007,No


In [0]:
from pyspark.sql.functions import col, to_date, concat, lit, when

cards_silver = (
    cards_bronze
    .withColumn("acct_open_date", to_date(concat(lit("01/"), col("acct_open_date")), "dd/MM/yyyy"))
    .withColumn(
        "card_on_dark_web",
        when(col("card_on_dark_web") == "Yes", True)
        .when(col("card_on_dark_web") == "No", False)
        .otherwise(None)
    )
    .dropDuplicates(["id"])
)

display(cards_silver.limit(10))
cards_silver.printSchema()

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,true,2,33900.0000,1991-01-01,2014,false
1,550,Mastercard,Credit,5278231764792292,06/2024,396,true,1,11600.0000,1994-01-01,2013,false
2,556,Mastercard,Debit,5889825928297675,09/2021,422,true,1,19948.0000,1995-01-01,2011,false
3,1937,Visa,Credit,4289888672554714,04/2020,736,true,2,16400.0000,1995-01-01,2015,false
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,true,2,19439.0000,1997-01-01,2007,false
5,619,Visa,Debit,4657824650820465,04/2024,245,true,2,21883.0000,1997-01-01,2012,false
6,1046,Amex,Credit,394584924614148,02/1999,302,true,2,9400.0000,1998-01-01,2011,false
7,511,Mastercard,Debit,5585238056278288,03/2005,749,true,1,9664.0000,1998-01-01,2011,false
8,1107,Mastercard,Credit,5462760953855576,09/2021,665,false,2,10300.0000,1998-01-01,2006,false
9,1046,Amex,Credit,357982644067712,09/2020,72,true,1,13000.0000,1999-01-01,2005,false


root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: date (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: boolean (nullable = true)



In [0]:
mcc_silver = mcc_bronze.dropDuplicates(["mcc_code"])

display(mcc_silver.limit(10))
mcc_silver.printSchema()

mcc_code,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
4900,"Utilities - Electric, Gas, Water, Sanitary"
5942,Book Stores
5814,Fast Food Restaurants
4829,Money Transfer
5311,Department Stores


root
 |-- mcc_code: string (nullable = true)
 |-- mcc_description: string (nullable = true)



In [0]:
users_bronze.printSchema()
display(users_bronze.limit(5))

root
 |-- id: string (nullable = true)
 |-- current_age: string (nullable = true)
 |-- retirement_age: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- birth_month: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- num_credit_cards: string (nullable = true)



id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [0]:
from pyspark.sql.functions import col, regexp_replace

users_silver = (
    users_bronze
    .withColumn("client_id", col("id").cast("int"))
    .withColumn("current_age", col("current_age").cast("int"))
    .withColumn("retirement_age", col("retirement_age").cast("int"))
    .withColumn("birth_year", col("birth_year").cast("int"))
    .withColumn("birth_month", col("birth_month").cast("int"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), r"[$,]", "").cast("double"))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), r"[$,]", "").cast("double"))
    .withColumn("total_debt", regexp_replace(col("total_debt"), r"[$,]", "").cast("double"))
    .withColumn("credit_score", col("credit_score").cast("int"))
    .withColumn("num_credit_cards", col("num_credit_cards").cast("int"))
    .drop("id")
    .dropDuplicates(["client_id"])
)

display(users_silver.limit(10))
users_silver.printSchema()

current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,client_id
53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5,825
53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5,1746
81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5,1718
63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4,708
43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1,1164
42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.0,41997.0,0.0,704,3,68
36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.0,51500.0,102286.0,672,3,1075
26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,26790.0,54623.0,114711.0,728,1,1711
81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.0,42509.0,2895.0,755,5,1116
34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.0,38190.0,81262.0,810,1,1752


root
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)
 |-- client_id: integer (nullable = true)



In [0]:
transactions_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.silver.transactions_data_silver")

cards_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.silver.cards_data_silver")

users_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.silver.users_data_silver")

mcc_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.silver.mcc_codes_silver")

In [0]:
print(f"Transactions bronze: {transactions_bronze.count()}")
print(f"Transactions silver: {transactions_silver.count()}")

Transactions bronze: 13305915
Transactions silver: 13305915
